# EUR/USD Volatility-Based Trading System
### Hedge Fund Quality Signal Generation, Backtesting & Risk Analysis

**Pipeline:**
1. Data & Feature Engineering
2. Stationarity Tests & Volatility Characteristics
3. Forecasting Models: EWMA · HAR-RV · ARRV · GARCH Family
4. Model Comparison & Selection
5. Volatility Regime Detection (K-Means)
6. Adaptive Signal Generation + Trend Filter
7. Full Backtest with Realistic Costs & Vol-Sizing
8. Performance Dashboard (Sharpe · Sortino · Calmar · Drawdown)
9. Monthly Returns Heatmap
10. Risk Analysis: VaR · CVaR · Monte Carlo
11. Parameter Sensitivity Heatmap
12. Walk-Forward Robustness Testing


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
from scipy.stats import norm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ── Professional Dark Theme ────────────────────────────────────────────────────
C = {
    'bg':     '#0D1117', 'panel':  '#161B22', 'border': '#30363D',
    'text':   '#E6EDF3', 'muted':  '#8B949E', 'blue':   '#58A6FF',
    'green':  '#3FB950', 'red':    '#F85149', 'orange': '#D29922',
    'purple': '#BC8CFF', 'teal':   '#39D353', 'yellow': '#E3B341',
}
plt.rcParams.update({
    'figure.facecolor': C['bg'],   'axes.facecolor':    C['panel'],
    'axes.edgecolor':   C['border'],'axes.labelcolor':  C['text'],
    'text.color':       C['text'], 'xtick.color':       C['muted'],
    'ytick.color':      C['muted'],'grid.color':        C['border'],
    'grid.alpha': 0.6,             'grid.linewidth':    0.6,
    'font.family': 'monospace',    'axes.titlesize':    13,
    'axes.labelsize': 11,          'figure.dpi':        120,
    'axes.spines.top': False,      'axes.spines.right': False,
    'legend.framealpha': 0.3,      'legend.edgecolor':  C['border'],
    'legend.facecolor': C['panel'],
})

fmt_pct = lambda x: f"{x:.2%}"
fmt_f2  = lambda x: f"{x:.2f}"
fmt_f3  = lambda x: f"{x:.3f}"

print("✓ Libraries loaded | Bloomberg dark theme active")


## 1  ·  Data Loading & Feature Engineering

In [ ]:
sp = pd.read_csv("Eurousd.csv", parse_dates=['Date'], index_col='Date')
sp = sp.sort_index()

# Core features
sp['log_return']    = np.log(sp['Close'] / sp['Close'].shift(1))
sp['realized_vol']  = sp['log_return'].rolling(21).std() * np.sqrt(252)
sp['squared_return']= sp['log_return'] ** 2
sp['abs_return']    = sp['log_return'].abs()
sp['atr_14']        = sp['log_return'].abs().rolling(14).mean()

# Trend filters
sp['ema_20']  = sp['Close'].ewm(span=20).mean()
sp['ema_50']  = sp['Close'].ewm(span=50).mean()
sp['ema_200'] = sp['Close'].ewm(span=200).mean()

# Momentum
sp['momentum_5']  = sp['Close'] / sp['Close'].shift(5)  - 1
sp['momentum_20'] = sp['Close'] / sp['Close'].shift(20) - 1

# Vol percentile rank
sp['vol_percentile'] = sp['realized_vol'].rolling(60).rank(pct=True) * 100

# HAR-RV components
sp['rv_weekly']  = sp['realized_vol'].rolling(5).mean()
sp['rv_monthly'] = sp['realized_vol'].rolling(22).mean()

sp.dropna(inplace=True)

print(f"✓ Data loaded: {len(sp):,} trading days")
print(f"  Period : {sp.index[0].date()} → {sp.index[-1].date()}")
print(f"  Columns: {len(sp.columns)}")
print(f"\n  Close      {sp['Close'].min():.4f} → {sp['Close'].max():.4f}")
print(f"  Avg RV     {sp['realized_vol'].mean():.4f}")
print(f"  Avg Return {sp['log_return'].mean():.6f}")
sp.tail(3)


## 2  ·  Volatility Analysis
### 2a  Stationarity Tests (ADF)

In [ ]:
def run_adf(series, name, sig=0.05):
    r = adfuller(series.dropna(), autolag='AIC')
    adf, p, lags, crit = r[0], r[1], r[2], r[4]
    stat = p < sig
    tag  = "STATIONARY    " if stat else "NON-STATIONARY"
    print(f"  [{tag}]  {name:<30}  ADF={adf:8.3f}  p={p:.4f}  lags={lags}")
    return {'Series': name, 'ADF Stat': round(adf,3), 'p-value': round(p,4),
            'Stationary': stat, '1% CV': round(crit['1%'],3)}

print("=" * 72)
print("  AUGMENTED DICKEY-FULLER STATIONARITY TESTS")
print("=" * 72)
tests = [
    (sp['Close'],          'Price (Levels)'),
    (sp['log_return'],     'Log Returns'),
    (sp['realized_vol'],   'Realized Volatility'),
    (sp['squared_return'], 'Squared Returns'),
    (sp['abs_return'],     'Absolute Returns'),
]
adf_rows = [run_adf(s, n) for s, n in tests]
print("=" * 72)
adf_df = pd.DataFrame(adf_rows).set_index('Series')

def _color_stat(col):
    return ['color: #3FB950' if v else 'color: #F85149' for v in adf_df['Stationary']]

adf_df.style.apply(_color_stat, subset=['Stationary'])


### 2b  Volatility Characteristics

In [ ]:
fig = plt.figure(figsize=(18, 13))
fig.suptitle('EUR/USD  ·  Volatility Characteristics', fontsize=16,
             color=C['text'], fontweight='bold', y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)

# 1. Realized vol time series
ax1 = fig.add_subplot(gs[0, :])
vol = sp['realized_vol']
ax1.fill_between(vol.index, vol, alpha=0.25, color=C['blue'])
ax1.plot(vol.index, vol, color=C['blue'], linewidth=0.7, label='Realized Vol (21d)')
p70 = vol.quantile(0.70); p30 = vol.quantile(0.30)
ax1.axhline(p70, color=C['red'],   linestyle='--', linewidth=1.2, label=f'70th pct  {p70:.4f}')
ax1.axhline(p30, color=C['green'], linestyle='--', linewidth=1.2, label=f'30th pct  {p30:.4f}')
ax1.set_title('Realized Volatility — Annualised', color=C['text'])
ax1.legend(fontsize=9, loc='upper right'); ax1.set_ylabel('Volatility')

# 2. Log returns
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(sp.index, sp['log_return'], color=C['teal'], linewidth=0.4, alpha=0.9)
ax2.axhline(0, color=C['muted'], linewidth=0.5)
ax2.set_title('Log Returns', color=C['text']); ax2.set_ylabel('Return')

# 3. Return distribution vs Normal
ax3 = fig.add_subplot(gs[1, 1])
ret = sp['log_return'].dropna()
ax3.hist(ret, bins=90, density=True, color=C['purple'], alpha=0.65, label='EUR/USD')
x = np.linspace(ret.min(), ret.max(), 300)
ax3.plot(x, norm.pdf(x, ret.mean(), ret.std()), color=C['orange'],
         linewidth=2, label='Normal Fit')
ax3.set_title(f'Return Distribution\nKurt={ret.kurt():.2f}  Skew={ret.skew():.2f}', color=C['text'])
ax3.legend(fontsize=8)

# 4. Vol distribution
ax4 = fig.add_subplot(gs[1, 2])
ax4.hist(vol.dropna(), bins=55, color=C['blue'], alpha=0.7, density=True)
ax4.set_title('Realized Vol Distribution', color=C['text']); ax4.set_xlabel('Volatility')

# 5. ACF of squared returns (ARCH effects)
ax5 = fig.add_subplot(gs[2, 0])
conf = 1.96 / np.sqrt(len(sp))
acf_sq = [sp['squared_return'].autocorr(i) for i in range(1, 31)]
ax5.bar(range(1, 31), acf_sq, color=C['blue'], alpha=0.75)
ax5.axhline(conf,  color=C['red'], linestyle='--', linewidth=1)
ax5.axhline(-conf, color=C['red'], linestyle='--', linewidth=1)
ax5.set_title('ACF — Squared Returns\n(ARCH effects)', color=C['text']); ax5.set_xlabel('Lag')

# 6. ACF of realized vol
ax6 = fig.add_subplot(gs[2, 1])
conf_rv = 1.96 / np.sqrt(len(vol.dropna()))
acf_rv = [vol.dropna().autocorr(i) for i in range(1, 31)]
ax6.bar(range(1, 31), acf_rv, color=C['green'], alpha=0.75)
ax6.axhline(conf_rv,  color=C['red'], linestyle='--', linewidth=1)
ax6.axhline(-conf_rv, color=C['red'], linestyle='--', linewidth=1)
ax6.set_title('ACF — Realized Vol\n(long memory)', color=C['text']); ax6.set_xlabel('Lag')

# 7. Q-Q plot
ax7 = fig.add_subplot(gs[2, 2])
(osm, osr), (slope, intercept, r) = stats.probplot(ret, dist='norm')
ax7.scatter(osm, osr, s=2, color=C['purple'], alpha=0.5)
ax7.plot(osm, slope*np.array(osm)+intercept, color=C['orange'],
         linewidth=1.5, label=f'R²={r**2:.3f}')
ax7.set_title('Q-Q Plot (Log Returns)', color=C['text'])
ax7.set_xlabel('Theoretical'); ax7.set_ylabel('Sample'); ax7.legend(fontsize=8)

for ax in [ax1,ax2,ax3,ax4,ax5,ax6,ax7]:
    ax.tick_params(colors=C['muted'])

plt.savefig('vol_characteristics.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()


## 3  ·  Forecasting Models
EWMA · HAR-RV · AutoReg Realised Vol (ARRV) · GARCH family


In [ ]:
rv  = sp['realized_vol']
ret = sp['log_return']

# ── EWMA ──────────────────────────────────────────────────────────────────────
def ewma_volatility(returns, lam=0.94):
    v = np.zeros(len(returns))
    v[0] = returns.iloc[0]**2
    for i in range(1, len(returns)):
        v[i] = lam*v[i-1] + (1-lam)*returns.iloc[i-1]**2
    return pd.Series(np.sqrt(v*252), index=returns.index)

# ── HAR-RV ─────────────────────────────────────────────────────────────────────
def fit_har_rv(rv_series):
    df = pd.DataFrame({'rv': rv_series,
                       'rv_d': rv_series.shift(1),
                       'rv_w': rv_series.shift(1).rolling(5).mean(),
                       'rv_m': rv_series.shift(1).rolling(22).mean()}).dropna()
    X = np.column_stack([np.ones(len(df)), df['rv_d'], df['rv_w'], df['rv_m']])
    y = df['rv'].values
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    yp   = X @ beta
    mae  = mean_absolute_error(y, yp)
    rmse = np.sqrt(mean_squared_error(y, yp))
    r2   = 1 - np.sum((y-yp)**2)/np.sum((y-y.mean())**2)
    fc   = pd.Series(np.nan, index=rv_series.index)
    fc.loc[df.index] = yp
    return fc, dict(mae=mae, rmse=rmse, r2=r2, beta=beta)

# ── ARRV ───────────────────────────────────────────────────────────────────────
def fit_arrv(rv_series, lags=5):
    clean    = rv_series.dropna()
    n_train  = int(len(clean)*0.80)
    train    = clean.iloc[:n_train]
    m        = AutoReg(train, lags=lags, old_names=False).fit()
    fitted   = m.fittedvalues
    history  = list(train)
    oos_preds= []
    for i in range(len(clean)-n_train):
        pm = AutoReg(history, lags=lags, old_names=False).fit()
        oos_preds.append(pm.predict(start=len(history), end=len(history))[0])
        history.append(clean.iloc[n_train+i])
    oos_actual = clean.iloc[n_train:]
    oos_pred   = pd.Series(oos_preds, index=oos_actual.index)
    mae  = mean_absolute_error(oos_actual, oos_pred)
    rmse = np.sqrt(mean_squared_error(oos_actual, oos_pred))
    r2   = 1 - np.sum((oos_actual.values-oos_pred.values)**2)/np.sum((oos_actual.values-oos_actual.mean())**2)
    fc   = pd.Series(np.nan, index=rv_series.index)
    fc.loc[fitted.index]   = fitted.values
    fc.loc[oos_pred.index] = oos_pred.values
    return fc, dict(mae=mae, rmse=rmse, r2=r2)

# ── GARCH family ───────────────────────────────────────────────────────────────
def fit_garch(returns_pct, vol_type, p=1, q=1):
    try:
        am  = arch_model(returns_pct.dropna(), vol=vol_type, p=p, q=q, dist='normal')
        res = am.fit(disp='off', show_warning=False)
        cv  = res.conditional_volatility / 100 * np.sqrt(252)
        act = sp['realized_vol'].reindex(cv.index).dropna()
        prd = cv.reindex(act.index).dropna()
        common = act.index.intersection(prd.index)
        mae  = mean_absolute_error(act.loc[common], prd.loc[common])
        rmse = np.sqrt(mean_squared_error(act.loc[common], prd.loc[common]))
        return cv, dict(mae=mae, rmse=rmse, aic=res.aic)
    except:
        return None, dict(mae=np.nan, rmse=np.nan, aic=np.nan)

print("=" * 65)
print("  RUNNING FORECASTING MODELS ...")
print("=" * 65)

ret_pct = ret * 100

ewma_fc = ewma_volatility(ret, lam=0.94)
ewma_mae = mean_absolute_error(rv.dropna(), ewma_fc.reindex(rv.dropna().index).dropna())
print(f"  EWMA           MAE: {ewma_mae:.6f}")

print("  HAR-RV ...", end=" ")
har_fc, har_m  = fit_har_rv(rv)
print(f"  MAE: {har_m['mae']:.6f}  RMSE: {har_m['rmse']:.6f}  In-sample R²: {har_m['r2']:.4f}")
print(f"         β_d={har_m['beta'][1]:.4f}  β_w={har_m['beta'][2]:.4f}  β_m={har_m['beta'][3]:.4f}")

print("  ARRV(5) ...", end=" ")
arrv_fc, arrv_m = fit_arrv(rv, lags=5)
print(f"  OOS MAE: {arrv_m['mae']:.6f}  OOS R²: {arrv_m['r2']:.4f}")

print("  GARCH   ...", end=" ")
garch_fc,   garch_m   = fit_garch(ret_pct, 'Garch')
print(f"  MAE: {garch_m['mae']:.6f}")

print("  EGARCH  ...", end=" ")
egarch_fc,  egarch_m  = fit_garch(ret_pct, 'EGARCH')
print(f"  MAE: {egarch_m['mae']:.6f}")

print("  APARCH  ...", end=" ")
aparch_fc,  aparch_m  = fit_garch(ret_pct, 'APARCH')
print(f"  MAE: {aparch_m['mae']:.6f}")

print("  FIGARCH ...", end=" ")
figarch_fc, figarch_m = fit_garch(ret_pct, 'FIGARCH')
print(f"  MAE: {figarch_m['mae']:.6f}")

print("=" * 65)
print("  Done.")


## 4  ·  Model Comparison & Selection

In [ ]:
model_df = pd.DataFrame({
    'Model':  ['EWMA','HAR-RV','ARRV(5)','GARCH','EGARCH','APARCH','FIGARCH'],
    'Type':   ['Stat','Stat','Stat','GARCH','GARCH','GARCH','GARCH'],
    'MAE':    [ewma_mae, har_m['mae'], arrv_m['mae'],
               garch_m['mae'], egarch_m['mae'], aparch_m['mae'], figarch_m['mae']],
    'OOS R²': [np.nan, har_m['r2'], arrv_m['r2'],
               np.nan, np.nan, np.nan, np.nan],
}).set_index('Model').round(6)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Volatility Model Comparison', fontsize=15, color=C['text'], fontweight='bold')

bar_colors = [C['blue'] if t=='Stat' else C['purple'] for t in model_df['Type']]
valid_mae = model_df['MAE'].dropna()

# MAE bar
axes[0].barh(valid_mae.index, valid_mae.values, color=bar_colors[:len(valid_mae)], alpha=0.85, edgecolor=C['border'])
axes[0].set_title('MAE  (lower = better)', color=C['text'])
for i, v in enumerate(valid_mae.values):
    axes[0].text(v*1.01, i, f'{v:.5f}', va='center', fontsize=8, color=C['muted'])

# R² bar
r2_df = model_df['OOS R²'].dropna()
r2_colors = [C['green'] if v > 0.5 else C['orange'] for v in r2_df]
axes[1].barh(r2_df.index, r2_df.values, color=r2_colors, alpha=0.85, edgecolor=C['border'])
axes[1].set_title('OOS R²  (higher = better)', color=C['text'])
axes[1].axvline(0, color=C['muted'], linewidth=0.8)
for i, v in enumerate(r2_df.values):
    axes[1].text(v*0.98, i, f'{v:.3f}', va='center', ha='right', fontsize=9,
                 color=C['text'], fontweight='bold')

# ARRV forecast vs actual (last 200 days)
rv_c   = rv.dropna()
arrv_c = arrv_fc.dropna().reindex(rv_c.index).dropna()
common = rv_c.index.intersection(arrv_c.index)[-200:]
axes[2].plot(common, rv_c.loc[common],   color=C['blue'],   linewidth=1,   label='Actual RV')
axes[2].plot(common, arrv_c.loc[common], color=C['orange'], linewidth=1,   label='ARRV Forecast', linestyle='--')
if garch_fc is not None:
    garch_c = garch_fc.reindex(common).dropna()
    if len(garch_c):
        axes[2].plot(garch_c.index, garch_c.values, color=C['purple'], linewidth=0.8,
                     label='GARCH', alpha=0.7, linestyle=':')
axes[2].set_title('ARRV vs Actual (last 200d)', color=C['text'])
axes[2].legend(fontsize=8); axes[2].set_ylabel('Volatility')

for ax in axes:
    ax.tick_params(colors=C['muted'])

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()
print(model_df.to_string())


## 5  ·  Volatility Regime Detection (K-Means)

In [ ]:
feat = pd.DataFrame({
    'vol':      sp['realized_vol'],
    'vol_5d':   sp['rv_weekly'],
    'vol_pct':  sp['vol_percentile']/100,
    'abs_ret':  sp['abs_return'],
}).dropna()

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(feat)

km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)
feat['regime_id'] = km.labels_

means = feat.groupby('regime_id')['vol'].mean().sort_values()
rmap  = {means.index[0]: 'LOW VOL', means.index[1]: 'MED VOL', means.index[2]: 'HIGH VOL'}
feat['regime'] = feat['regime_id'].map(rmap)

sp['regime'] = feat['regime'].reindex(sp.index)
sp['regime'].ffill(inplace=True)

RCOLS = {'LOW VOL': C['green'], 'MED VOL': C['orange'], 'HIGH VOL': C['red']}

stats_tbl = feat.groupby('regime').agg(
    Days=('vol','count'), Mean_Vol=('vol','mean'), Median_Vol=('vol','median'), Std=('vol','std')
).round(4)
print(stats_tbl.to_string())

fig, axes = plt.subplots(3, 1, figsize=(18, 12), gridspec_kw={'height_ratios':[2,1,1]})
fig.suptitle('EUR/USD  ·  Volatility Regime Analysis', fontsize=16, color=C['text'], fontweight='bold')

ax1 = axes[0]
ax1.plot(sp.index, sp['Close'], color=C['text'], linewidth=0.7, alpha=0.9, zorder=3)
ax1.plot(sp.index, sp['ema_50'], color=C['blue'], linewidth=1, alpha=0.7, label='EMA-50', zorder=2)
ymin, ymax = sp['Close'].min(), sp['Close'].max()
for r, col in RCOLS.items():
    mask = sp['regime'] == r
    ax1.fill_between(sp.index, ymin, ymax, where=mask, alpha=0.10, color=col, label=r)
ax1.set_title('EUR/USD Price  ·  Regime Background', color=C['text'])
ax1.legend(loc='upper left', fontsize=9, ncol=4)
ax1.set_ylabel('Price')

ax2 = axes[1]
for r, col in RCOLS.items():
    mask = sp['regime'] == r
    ax2.fill_between(sp.index, 0, sp['realized_vol'], where=mask, alpha=0.65, color=col)
ax2.plot(sp.index, sp['realized_vol'], color=C['muted'], linewidth=0.4, alpha=0.6)
ax2.set_title('Realized Volatility  ·  Colour = Regime', color=C['text']); ax2.set_ylabel('Vol')

ax3 = axes[2]
ax3.plot(sp.index, sp['vol_percentile'], color=C['purple'], linewidth=0.7)
ax3.axhline(70, color=C['red'],   linestyle='--', linewidth=1, label='Entry zone (>70th)')
ax3.axhline(30, color=C['green'], linestyle='--', linewidth=1, label='Exit zone (<30th)')
ax3.fill_between(sp.index, 70, 100, alpha=0.1, color=C['red'])
ax3.fill_between(sp.index, 0,  30,  alpha=0.1, color=C['green'])
ax3.set_title('Vol Percentile Rank (60-day rolling)', color=C['text'])
ax3.set_ylabel('Percentile'); ax3.set_ylim(0, 100); ax3.legend(fontsize=9)

for ax in axes:
    ax.tick_params(colors=C['muted'])

plt.tight_layout()
plt.savefig('regime_analysis.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()


## 6  ·  Volatility Trading System

**Signal logic**
- **Entry**: realized vol > adaptive 70th percentile (60-day rolling window)
- **Direction**: filtered by EMA-50 trend — long above, short below
- **Regime gate**: skip entries when regime = LOW VOL
- **Exit**: vol drops below adaptive 30th percentile OR trend flips
- **Sizing**: inverse-vol scaling (target-vol approach) — smaller size when vol is high
- **Costs**: 0.5 pip per trade (realistic EUR/USD)


In [ ]:
class VolatilityTradingSystem:

    def __init__(self, data):
        self.data        = data.copy()
        self.signals_df  = None
        self.results_df  = None

    def generate_signals(self, entry_pct=70, exit_pct=30, roll_window=60,
                         trend_filter=True, ema_span=50, regime_filter=True):
        df = self.data.copy()
        df['vol_pct']   = df['realized_vol'].rolling(roll_window).rank(pct=True) * 100
        df['ema']       = df['Close'].ewm(span=ema_span).mean()
        df['trend']     = np.where(df['Close'] > df['ema'], 1, -1)

        signal   = np.zeros(len(df))
        position = 0

        for i in range(roll_window, len(df)):
            vp = df['vol_pct'].iloc[i]
            tr = df['trend'].iloc[i]
            rg = df['regime'].iloc[i] if 'regime' in df.columns else 'MED VOL'

            if position == 0:
                if vp >= entry_pct:
                    if regime_filter and rg == 'LOW VOL':
                        pass
                    else:
                        position = int(tr) if trend_filter else 1
            else:
                if vp <= exit_pct:
                    position = 0
                elif trend_filter and np.sign(tr) != np.sign(position):
                    position = 0

            signal[i] = position

        df['signal']          = signal
        df['position_change'] = np.abs(np.diff(signal, prepend=0)) > 0
        self.signals_df       = df
        return df

    def backtest(self, leverage=1.0, transaction_cost=0.00005,
                 use_vol_sizing=True, target_vol=None):
        df = self.signals_df.copy()

        if use_vol_sizing:
            tv  = target_vol or df['realized_vol'].mean()
            scl = (tv / df['realized_vol']).clip(0.25, 4.0)
            df['position_size'] = df['signal'] * scl * leverage
        else:
            df['position_size'] = df['signal'] * leverage

        df['price_ret'] = df['Close'].pct_change()
        df['gross_ret'] = df['position_size'].shift(1) * df['price_ret']
        df['cost']      = df['position_change'].shift(1).fillna(0) * transaction_cost
        df['net_ret']   = df['gross_ret'] - df['cost']

        df['equity']     = (1 + df['net_ret'].fillna(0)).cumprod()
        df['bh_equity']  = (1 + df['price_ret'].fillna(0)).cumprod()
        df['peak']       = df['equity'].cummax()
        df['drawdown']   = df['equity'] / df['peak'] - 1
        df['bh_peak']    = df['bh_equity'].cummax()
        df['bh_dd']      = df['bh_equity'] / df['bh_peak'] - 1

        self.results_df = df
        return df

    def get_metrics(self):
        df  = self.results_df
        ret = df['net_ret'].dropna()
        eq  = df['equity'].dropna()
        dd  = df['drawdown'].dropna()
        ann = 252

        ann_ret = eq.iloc[-1] ** (ann/len(eq)) - 1 if len(eq) > 1 else 0
        ann_vol = ret.std() * np.sqrt(ann)
        sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0

        neg     = ret[ret < 0]
        sortino = ann_ret / (neg.std()*np.sqrt(ann)) if len(neg) > 1 else 0

        max_dd  = dd.min()
        calmar  = ann_ret / abs(max_dd) if max_dd != 0 else 0
        win_r   = (ret > 0).mean()
        pf      = ret[ret>0].sum() / abs(ret[ret<0].sum()) if len(ret[ret<0])>0 else np.inf
        trades  = int(df['position_change'].sum())
        var95   = np.percentile(ret, 5)
        cvar95  = ret[ret <= var95].mean()

        return {
            'Annual Return':    fmt_pct(ann_ret),
            'Annual Volatility':fmt_pct(ann_vol),
            'Total Return':     fmt_pct(eq.iloc[-1]-1),
            'Sharpe Ratio':     fmt_f3(sharpe),
            'Sortino Ratio':    fmt_f3(sortino),
            'Calmar Ratio':     fmt_f3(calmar),
            'Max Drawdown':     fmt_pct(max_dd),
            'VaR 95%':          fmt_pct(var95),
            'CVaR 95%':         fmt_pct(cvar95),
            'Win Rate':         fmt_pct(win_r),
            'Profit Factor':    fmt_f2(pf),
            'Total Trades':     str(trades),
            'Skewness':         fmt_f3(ret.skew()),
            'Kurtosis':         fmt_f3(ret.kurt()),
        }

print("✓ VolatilityTradingSystem class ready")


### 6a  Baseline Backtest

In [ ]:
ts = VolatilityTradingSystem(sp.dropna())

ts.generate_signals(entry_pct=70, exit_pct=30, roll_window=60,
                    trend_filter=True, ema_span=50, regime_filter=True)

ts.backtest(leverage=1.0, transaction_cost=0.00005, use_vol_sizing=True)

m = ts.get_metrics()
print()
print("=" * 52)
print("  PERFORMANCE METRICS")
print("=" * 52)
sections = {
    'RETURNS':      ['Annual Return','Annual Volatility','Total Return'],
    'RISK-ADJUSTED':['Sharpe Ratio','Sortino Ratio','Calmar Ratio'],
    'RISK':         ['Max Drawdown','VaR 95%','CVaR 95%'],
    'TRADING':      ['Win Rate','Profit Factor','Total Trades'],
    'DISTRIBUTION': ['Skewness','Kurtosis'],
}
for sec, keys in sections.items():
    print(f"\n  ── {sec} ──")
    for k in keys:
        print(f"    {k:<20} {m[k]}")
print("=" * 52)


## 7  ·  Performance Dashboard

In [ ]:
df = ts.results_df

fig = plt.figure(figsize=(18, 15))
fig.suptitle('EUR/USD Volatility Trading System  ·  Performance Dashboard',
             fontsize=16, color=C['text'], fontweight='bold', y=0.99)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.48, wspace=0.35)

# 1. Equity curves (full width)
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df.index, df['equity'],    color=C['green'], linewidth=1.5, label='Strategy',    zorder=3)
ax1.plot(df.index, df['bh_equity'], color=C['blue'],  linewidth=1.0, label='Buy & Hold',  alpha=0.7, zorder=2)
ax1.fill_between(df.index, 1, df['equity'], where=df['equity'] >= 1, alpha=0.12, color=C['green'])
ax1.fill_between(df.index, 1, df['equity'], where=df['equity'] < 1,  alpha=0.12, color=C['red'])
ax1.axhline(1, color=C['muted'], linewidth=0.8, linestyle='--')
ax1.set_title('Equity Curve  ·  Strategy vs Buy & Hold', color=C['text'])
ax1.set_ylabel('Portfolio Value (rebased 1.0)'); ax1.legend(loc='upper left', fontsize=10)

# 2. Drawdown (full width)
ax2 = fig.add_subplot(gs[1, :])
ax2.fill_between(df.index, df['drawdown']*100, 0, alpha=0.65, color=C['red'])
ax2.plot(df.index,       df['drawdown']*100,       color=C['red'],  linewidth=0.7, label='Strategy DD')
ax2.fill_between(df.index, df['bh_dd']*100, 0, alpha=0.25, color=C['blue'])
ax2.set_title('Drawdown (%)', color=C['text']); ax2.set_ylabel('Drawdown (%)')
patch_bh = plt.matplotlib.patches.Patch(color=C['blue'], alpha=0.5, label='Buy & Hold DD')
ax2.legend(handles=[ax2.lines[0], patch_bh], fontsize=9)

# 3. Rolling Sharpe (63d)
ax3 = fig.add_subplot(gs[2, 0])
roll_sh = (df['net_ret'].rolling(63).mean() / df['net_ret'].rolling(63).std()) * np.sqrt(252)
ax3.plot(df.index, roll_sh, color=C['purple'], linewidth=0.8)
ax3.axhline(0, color=C['muted'], linewidth=0.7, linestyle='--')
ax3.axhline(1, color=C['green'], linewidth=0.7, linestyle=':', alpha=0.7)
ax3.fill_between(df.index, roll_sh, 0, where=roll_sh>=0, alpha=0.18, color=C['green'])
ax3.fill_between(df.index, roll_sh, 0, where=roll_sh<0,  alpha=0.18, color=C['red'])
ax3.set_title('Rolling 63d Sharpe', color=C['text']); ax3.set_ylabel('Sharpe')

# 4. Return distribution
ax4 = fig.add_subplot(gs[2, 1])
sr = df['net_ret'].dropna(); bhr = df['price_ret'].dropna()
ax4.hist(sr,  bins=80, density=True, color=C['green'], alpha=0.55, label='Strategy')
ax4.hist(bhr, bins=80, density=True, color=C['blue'],  alpha=0.35, label='Buy & Hold')
xr = np.linspace(sr.quantile(0.01), sr.quantile(0.99), 200)
ax4.plot(xr, norm.pdf(xr, sr.mean(), sr.std()), color=C['orange'], linewidth=1.5,
         linestyle='--', label='Normal')
var95_val = np.percentile(sr, 5)
ax4.axvline(var95_val, color=C['red'], linewidth=1.5, label=f'VaR(95%): {var95_val:.4f}')
ax4.set_title('Return Distribution', color=C['text']); ax4.legend(fontsize=8)

# 5. Signal (last 300d)
ax5 = fig.add_subplot(gs[2, 2])
tail = df.iloc[-300:]
ax5.plot(tail.index, tail['signal'], color=C['yellow'], linewidth=0.8, alpha=0.8)
ax5.fill_between(tail.index, tail['signal'], 0,
                 where=tail['signal']>0, alpha=0.3, color=C['green'])
ax5.fill_between(tail.index, tail['signal'], 0,
                 where=tail['signal']<0, alpha=0.3, color=C['red'])
ax5.axhline(0, color=C['muted'], linewidth=0.5)
ax5.set_title('Signal  (last 300 days)', color=C['text'])
ax5.set_ylim(-1.5, 1.5); ax5.set_yticks([-1, 0, 1])
ax5.set_yticklabels(['Short','Flat','Long'])

# 6. Position size
ax6 = fig.add_subplot(gs[3, :])
ax6.plot(df.index, df['position_size'], color=C['teal'], linewidth=0.55, alpha=0.8)
ax6.fill_between(df.index, df['position_size'], 0,
                 where=df['position_size']>0, alpha=0.18, color=C['green'])
ax6.fill_between(df.index, df['position_size'], 0,
                 where=df['position_size']<0, alpha=0.18, color=C['red'])
ax6.axhline(0, color=C['muted'], linewidth=0.5)
ax6.set_title('Position Size  ·  Vol-Adjusted (smaller when vol is high)', color=C['text'])
ax6.set_ylabel('Position')

for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
    ax.tick_params(colors=C['muted'])

plt.savefig('performance_dashboard.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()


## 8  ·  Monthly Returns Heatmap

In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly = df['net_ret'].resample('ME').apply(lambda x: (1+x).prod()-1)
mpivot  = pd.DataFrame({'Year': monthly.index.year,
                         'Month': monthly.index.month,
                         'Return': monthly.values})             .pivot(index='Year', columns='Month', values='Return')

# Map month numbers to names dynamically (handles partial years)
mpivot.columns = [MONTH_NAMES[m-1] for m in mpivot.columns]
annual = mpivot.apply(lambda r: (1+r.dropna()).prod()-1, axis=1)
mpivot['Annual'] = annual

fig, ax = plt.subplots(figsize=(max(10, len(mpivot.columns)*1.4), max(4, len(mpivot)*0.75)))
fig.suptitle('Monthly Returns Heatmap  · Volatility Strategy',
             fontsize=15, color=C['text'], fontweight='bold')

cmap = LinearSegmentedColormap.from_list('rg', [C['red'], C['bg'], C['green']])
vals = mpivot.values.astype(float)
finite = vals[np.isfinite(vals)]
vmax = np.percentile(np.abs(finite), 95) if len(finite) else 0.01
im   = ax.imshow(vals, cmap=cmap, vmin=-vmax, vmax=vmax, aspect='auto')

for i in range(len(mpivot)):
    for j in range(len(mpivot.columns)):
        v = mpivot.values[i, j]
        if not np.isnan(v):
            col = 'white' if abs(v) > vmax*0.45 else C['muted']
            ax.text(j, i, f'{v*100:.1f}%', ha='center', va='center', fontsize=8, color=col)

ax.set_xticks(range(len(mpivot.columns)))
ax.set_xticklabels(mpivot.columns, color=C['text'], fontsize=10)
ax.set_yticks(range(len(mpivot.index)))
ax.set_yticklabels(mpivot.index, color=C['text'], fontsize=10)
ax.set_title('Green = Profit  · Red = Loss  · Last column = Annual', color=C['text'], pad=12)
plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01, label='Return')

plt.tight_layout()
plt.savefig('monthly_heatmap.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()

print("
Annual Returns:")
for yr, r in annual.items():
    sign = '+' if r >= 0 else ''
    bar  = '#' * min(30, int(abs(r)*300))
    print(f"  {yr}  {sign}{r:.2%}  {bar}")


## 9  ·  Risk Analysis  ·  VaR  ·  CVaR  ·  Monte Carlo

In [ ]:
ret = df['net_ret'].dropna()
bhr = df['price_ret'].dropna()

# Use aligned series (same index as df) for rolling/plot operations
ret_aligned = df['net_ret'].fillna(0)
bhr_aligned = df['price_ret'].fillna(0)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Risk Analysis Suite', fontsize=15, color=C['text'], fontweight='bold')

# 1. VaR / CVaR histogram
ax1 = axes[0, 0]
ax1.hist(ret, bins=100, density=True, color=C['blue'], alpha=0.6, label='Daily Returns')
for conf, col, lab in [(0.10, C['yellow'], 'VaR 90%'),
                       (0.05, C['orange'], 'VaR 95%'),
                       (0.01, C['red'],    'VaR 99%')]:
    v = np.percentile(ret, conf*100)
    ax1.axvline(v, color=col, linewidth=1.5, linestyle='--', label=f'{lab}: {v:.4f}')
cvar = ret[ret <= np.percentile(ret, 5)].mean()
ax1.axvline(cvar, color=C['purple'], linewidth=1.5, label=f'CVaR 95%: {cvar:.4f}')
ax1.set_title('Historical VaR & CVaR', color=C['text']); ax1.legend(fontsize=8)

# 2. Rolling 63d VaR & CVaR (use aligned series so index matches df)
ax2 = axes[0, 1]
rv63  = ret_aligned.rolling(63).quantile(0.05)

def rolling_cvar(x):
    thresh = np.percentile(x, 5)
    tail = x[x <= thresh]
    return tail.mean() if len(tail) > 0 else np.nan

cvr63 = ret_aligned.rolling(63).apply(rolling_cvar, raw=True)

ax2.plot(df.index, rv63 * 100,  color=C['orange'], linewidth=0.8, label='Rolling VaR 95%')
ax2.plot(df.index, cvr63 * 100, color=C['red'],    linewidth=0.8, label='Rolling CVaR 95%')
ax2.fill_between(df.index, rv63 * 100, 0, alpha=0.12, color=C['red'])
ax2.axhline(0, color=C['muted'], linewidth=0.5)
ax2.set_title('Rolling 63d VaR & CVaR (%)', color=C['text'])
ax2.set_ylabel('Return (%)'); ax2.legend(fontsize=9)

# 3. Drawdown distribution
ax3 = axes[1, 0]
dd_pct = df['drawdown'].dropna() * 100
ax3.hist(dd_pct, bins=60, color=C['red'], alpha=0.7, density=True)
ax3.axvline(dd_pct.mean(), color=C['orange'], linewidth=1.5, linestyle='--',
            label=f'Mean DD: {dd_pct.mean():.1f}%')
ax3.axvline(dd_pct.min(),  color=C['red'],    linewidth=1.5,
            label=f'Max DD:  {dd_pct.min():.1f}%')
ax3.set_title('Drawdown Distribution', color=C['text'])
ax3.set_xlabel('Drawdown (%)'); ax3.legend(fontsize=9)

# 4. Strategy vs B&H comparison bar chart
ax4 = axes[1, 1]
comp = pd.DataFrame({
    'Strategy':   [ret.mean()*252,
                   ret.std()*np.sqrt(252),
                   ret.mean()/ret.std()*np.sqrt(252) if ret.std()>0 else 0,
                   df['drawdown'].min(),
                   np.percentile(ret, 5)],
    'Buy & Hold': [bhr.mean()*252,
                   bhr.std()*np.sqrt(252),
                   bhr.mean()/bhr.std()*np.sqrt(252) if bhr.std()>0 else 0,
                   df['bh_dd'].min(),
                   np.percentile(bhr, 5)],
}, index=['Ann Return', 'Ann Vol', 'Sharpe', 'Max DD', 'VaR 95%'])

x = np.arange(len(comp))
ax4.bar(x-0.18, comp['Strategy'],   0.35, color=C['green'], alpha=0.8, label='Strategy')
ax4.bar(x+0.18, comp['Buy & Hold'], 0.35, color=C['blue'],  alpha=0.8, label='Buy & Hold')
ax4.set_xticks(x)
ax4.set_xticklabels(comp.index, rotation=20, ha='right', fontsize=9)
ax4.axhline(0, color=C['muted'], linewidth=0.5)
ax4.set_title('Strategy vs Buy & Hold', color=C['text']); ax4.legend(fontsize=9)

for ax in axes.flatten():
    ax.tick_params(colors=C['muted'])

plt.tight_layout()
plt.savefig('risk_analysis.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()


In [ ]:
N_SIM = 1000
N_DAYS = 252
mu    = ret.mean()
sigma = ret.std()

np.random.seed(42)
sim_ret = np.random.normal(mu, sigma, (N_SIM, N_DAYS))
sim_eq  = np.cumprod(1 + sim_ret, axis=1)
final   = sim_eq[:, -1]
ann_ret = final ** (252/N_DAYS) - 1

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Monte Carlo  ·  {N_SIM} paths  ·  {N_DAYS} days',
             fontsize=14, color=C['text'], fontweight='bold')

# Paths
ax1 = axes[0]
for i in range(200):
    col = C['green'] if sim_eq[i,-1]>1 else C['red']
    ax1.plot(sim_eq[i], color=col, alpha=0.04, linewidth=0.4)
x = np.arange(N_DAYS)
for pct, col, lbl in [(5,'#F85149','5th'),(25,'#D29922','25th'),
                       (50,'#58A6FF','Median'),(75,'#D29922','75th'),(95,'#3FB950','95th')]:
    band = np.percentile(sim_eq, pct, axis=0)
    ax1.plot(x, band, color=col, linewidth=1.5 if pct==50 else 0.8, label=lbl)
ax1.fill_between(x, np.percentile(sim_eq,5,axis=0), np.percentile(sim_eq,95,axis=0),
                 alpha=0.12, color=C['blue'])
ax1.axhline(1, color=C['muted'], linewidth=0.8, linestyle='--')
ax1.set_title('Simulated Equity Paths', color=C['text'])
ax1.set_xlabel('Days'); ax1.set_ylabel('Portfolio Value'); ax1.legend(fontsize=8)

# Final value dist
ax2 = axes[1]
ax2.hist(final, bins=60, color=C['blue'], alpha=0.7, density=True)
for pct, col, lbl in [(5,C['red'],'5th'),(50,C['green'],'Median'),(95,C['teal'],'95th')]:
    v = np.percentile(final, pct)
    ax2.axvline(v, color=col, linewidth=1.5, label=f'{lbl}: {v:.3f}')
ax2.axvline(1, color=C['muted'], linewidth=1, linestyle='--', label='Break-even')
ax2.set_title(f'Final Value Distribution\nP(profit): {(final>1).mean():.1%}', color=C['text'])
ax2.legend(fontsize=8)

# Stats card
ax3 = axes[2]
ax3.axis('off')
rows = [
    ('Expected Annual Return',  fmt_pct(ann_ret.mean())),
    ('Median Annual Return',    fmt_pct(np.median(ann_ret))),
    ('5th pct Annual Return',   fmt_pct(np.percentile(ann_ret,5))),
    ('95th pct Annual Return',  fmt_pct(np.percentile(ann_ret,95))),
    ('─────────────────────', ''),
    ('P(profit > 0%)',           fmt_pct((final>1).mean())),
    ('P(profit > 10%)',          fmt_pct((ann_ret>0.10).mean())),
    ('P(loss > 10%)',            fmt_pct((ann_ret<-0.10).mean())),
    ('─────────────────────', ''),
    ('Worst case (1st pct)',     fmt_pct(np.percentile(ann_ret,1))),
    ('Best case  (99th pct)',    fmt_pct(np.percentile(ann_ret,99))),
]
y = 0.95
for k, v in rows:
    if v == '':
        ax3.text(0.05, y, k, transform=ax3.transAxes, fontsize=8, color=C['border'])
    else:
        ax3.text(0.05, y, k, transform=ax3.transAxes, fontsize=9, color=C['muted'])
        ax3.text(0.75, y, v, transform=ax3.transAxes, fontsize=9, color=C['text'], fontweight='bold')
    y -= 0.075

ax3.set_facecolor(C['panel'])
ax3.set_title('Monte Carlo Statistics', color=C['text'], pad=10)

for ax in [ax1, ax2]:
    ax.tick_params(colors=C['muted'])

plt.tight_layout()
plt.savefig('monte_carlo.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()


## 10  ·  Parameter Sensitivity Heatmap

In [ ]:
# NOTE: This cell runs ~30 backtests — takes ~1-2 minutes
entry_range = [55, 60, 65, 70, 75, 80]
exit_range  = [20, 25, 30, 35, 40]

sharpe_grid = np.full((len(entry_range), len(exit_range)), np.nan)
return_grid = np.full((len(entry_range), len(exit_range)), np.nan)

ts_p = VolatilityTradingSystem(sp.dropna())

for i, ep in enumerate(entry_range):
    for j, xp in enumerate(exit_range):
        if ep <= xp:
            continue
        try:
            ts_p.generate_signals(entry_pct=ep, exit_pct=xp, roll_window=60,
                                  trend_filter=True, ema_span=50, regime_filter=True)
            ts_p.backtest(leverage=1.0, transaction_cost=0.00005, use_vol_sizing=True)
            pm = ts_p.get_metrics()
            sharpe_grid[i, j] = float(pm['Sharpe Ratio'])
            return_grid[i, j] = float(pm['Annual Return'].replace('%',''))/100
        except:
            pass
    print(f"  Entry {ep}th pct  ✓")

best = np.unravel_index(np.nanargmax(sharpe_grid), sharpe_grid.shape)
print(f"\n  Best Sharpe: {sharpe_grid[best]:.3f}"
      f"  →  Entry={entry_range[best[0]]}th  Exit={exit_range[best[1]]}th")

cmap_gr = LinearSegmentedColormap.from_list('gr', [C['red'], C['bg'], C['green']])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Parameter Sensitivity  ·  Entry vs Exit Percentile',
             fontsize=14, color=C['text'], fontweight='bold')

for ax, grid, title, fmt in zip(axes,
                                  [sharpe_grid, return_grid],
                                  ['Sharpe Ratio','Annual Return'],
                                  ['.2f', '.1%']):
    lo = np.nanpercentile(grid, 10); hi = np.nanpercentile(grid, 90)
    im = ax.imshow(grid, cmap=cmap_gr, vmin=lo, vmax=hi, aspect='auto')
    for i in range(len(entry_range)):
        for j in range(len(exit_range)):
            v = grid[i,j]
            if not np.isnan(v):
                txt = f'{v:{fmt}}'
                col = 'white' if abs(v-lo)/(hi-lo+1e-9) > 0.5 else C['muted']
                ax.text(j, i, txt, ha='center', va='center', fontsize=9, color=col)
    ax.set_xticks(range(len(exit_range)))
    ax.set_yticks(range(len(entry_range)))
    ax.set_xticklabels([f'{x}th' for x in exit_range],  color=C['text'])
    ax.set_yticklabels([f'{x}th' for x in entry_range], color=C['text'])
    ax.set_xlabel('Exit Percentile',  color=C['text'])
    ax.set_ylabel('Entry Percentile', color=C['text'])
    ax.set_title(title, color=C['text'])
    # Mark best
    ax.add_patch(plt.Rectangle((best[1]-0.5, best[0]-0.5), 1, 1,
                                fill=False, edgecolor=C['yellow'], linewidth=2))
    plt.colorbar(im, ax=ax, fraction=0.04)

plt.tight_layout()
plt.savefig('parameter_sensitivity.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()

# Re-run with best params
BEST_ENTRY = entry_range[best[0]]
BEST_EXIT  = exit_range[best[1]]


## 11  ·  Walk-Forward Robustness Test

In [ ]:
data_wf = sp.dropna()
n       = len(data_wf)
t_end   = int(n * 0.60)
v_end   = int(n * 0.80)

splits = [
    ('Train      (60%)', data_wf.iloc[:t_end],      C['blue']),
    ('Validation (20%)', data_wf.iloc[t_end:v_end], C['orange']),
    ('Test       (20%)', data_wf.iloc[v_end:],      C['green']),
]

fig, axes = plt.subplots(3, 1, figsize=(18, 13), sharex=False)
fig.suptitle('Walk-Forward Analysis  · Train / Validation / Test',
             fontsize=15, color=C['text'], fontweight='bold')

all_m = {}
for idx, (name, sdata, col) in enumerate(splits):
    ts_wf = VolatilityTradingSystem(sdata)
    ts_wf.generate_signals(entry_pct=BEST_ENTRY, exit_pct=BEST_EXIT,
                           roll_window=60, trend_filter=True, regime_filter=True)
    ts_wf.backtest(leverage=1.0, transaction_cost=0.00005, use_vol_sizing=True)
    pm = ts_wf.get_metrics()
    all_m[name.strip()] = pm

    dr = ts_wf.results_df

    axes[idx].plot(dr.index, dr['equity'].values.astype(float),
                   color=col, linewidth=1.5, label=f'{name} - Strategy')
    axes[idx].plot(dr.index, dr['bh_equity'].values.astype(float),
                   color=C['muted'], linewidth=0.9, linestyle='--',
                   label='Buy & Hold', alpha=0.7)

    # Coerce drawdown to clean float array before fill_between
    dd_fill = pd.to_numeric(dr['drawdown'], errors='coerce').fillna(0).values.astype(float)
    axes[idx].fill_between(dr.index, dd_fill * 5 + 1, 1.0, alpha=0.12, color=C['red'])

    axes[idx].axhline(1, color=C['muted'], linewidth=0.6, linestyle=':')
    info = (f"Sharpe: {pm['Sharpe Ratio']}  |  "
            f"Return: {pm['Annual Return']}  |  "
            f"Max DD: {pm['Max Drawdown']}  |  "
            f"Win: {pm['Win Rate']}  |  n={len(sdata):,}d")
    axes[idx].set_title(f'{name}  ·  {info}', color=C['text'])
    axes[idx].legend(loc='upper left', fontsize=9)
    axes[idx].set_ylabel('Portfolio Value')
    axes[idx].tick_params(colors=C['muted'])

plt.tight_layout()
plt.savefig('walk_forward.png', dpi=140, bbox_inches='tight', facecolor=C['bg'])
plt.show()

# Comparison table
keys = ['Annual Return', 'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown', 'Win Rate', 'Profit Factor']
print(f"
{'Metric':<22}", end='')
for name, _, _ in splits:
    print(f" {name.split('(')[0].strip():>14}", end='')
print()
print("-" * 66)
for k in keys:
    print(f"  {k:<20}", end='')
    for name, _, _ in splits:
        nm = name.strip()
        print(f" {all_m[nm][k]:>14}", end='')
    print()


## 12  ·  Final Summary

In [ ]:
# Re-run full-period backtest with best params
ts_final = VolatilityTradingSystem(sp.dropna())
ts_final.generate_signals(entry_pct=BEST_ENTRY, exit_pct=BEST_EXIT,
                          roll_window=60, trend_filter=True, regime_filter=True)
ts_final.backtest(leverage=1.0, transaction_cost=0.00005, use_vol_sizing=True)
fm = ts_final.get_metrics()

W = 66
print("╔" + "═"*W + "╗")
print("║" + "  EUR/USD VOLATILITY TRADING SYSTEM — SUMMARY".center(W) + "║")
print("╠" + "═"*W + "╣")
print("║" + f"  Best Forecasting Model  : ARRV(5)  OOS R² = {arrv_m['r2']:.4f}".ljust(W) + "║")
print("║" + f"  Signal Entry            : Vol > {BEST_ENTRY}th percentile (adaptive 60d)".ljust(W) + "║")
print("║" + f"  Signal Exit             : Vol < {BEST_EXIT}th percentile OR trend flip".ljust(W) + "║")
print("║" + "  Direction Filter        : EMA-50 trend  (long/short)".ljust(W) + "║")
print("║" + "  Regime Gate             : No entries in LOW VOL regime".ljust(W) + "║")
print("║" + "  Position Sizing         : Inverse-vol scaling (target-vol)".ljust(W) + "║")
print("║" + "  Transaction Cost        : 0.5 pip per trade (realistic)".ljust(W) + "║")
print("╠" + "═"*W + "╣")
print("║" + "  FULL-PERIOD PERFORMANCE".ljust(W) + "║")
print("╠" + "═"*W + "╣")
pairs = [('Annual Return','Sharpe Ratio'),('Annual Volatility','Sortino Ratio'),
         ('Max Drawdown','Calmar Ratio'),  ('Win Rate','Profit Factor'),
         ('VaR 95%','CVaR 95%'),           ('Total Trades','Skewness')]
for k1, k2 in pairs:
    line = f"  {k1:<22} {fm.get(k1,'—'):<14}   {k2:<22} {fm.get(k2,'—')}"
    print("║" + line.ljust(W) + "║")
print("╚" + "═"*W + "╝")

print("\n  Saved charts:")
for f in ['vol_characteristics','model_comparison','regime_analysis',
          'performance_dashboard','monthly_heatmap','risk_analysis',
          'monte_carlo','parameter_sensitivity','walk_forward']:
    print(f"    ✓  {f}.png")
